In [4]:
import sys
!{sys.executable} -m pip uninstall torch torchvision -y
!{sys.executable} -m pip install torch torchvision

Found existing installation: torch 2.9.0

You can safely remove it manually.



Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.24.0
Uninstalling torchvision-0.24.0:
  Successfully uninstalled torchvision-0.24.0
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/114.5 MB 8.3 MB/s eta 0:00:14
   -- ------------------------------------- 8.4/114.5 MB 28.8 MB/s eta 0:00:04
   ------ --------------------------------- 19.1/114.5 MB 46.4 MB/s eta 0:00:03
   ----------- ---------------------------- 33.0/114.5 MB 45.6 MB/s eta 0:00:02
   --------------- ------------------------ 45.4/114.5 MB 50.6 MB/s eta 0:00:02
   -------------------- ------------------- 59.5/114.5 MB 51.9 MB/s eta 0:00:02
   ------------------------ --------------- 69.7/114.5 MB 51.7 MB/s eta 0:00:01
   ----------------------------- ---------- 85.5/114.5 MB 54.5 MB/s eta 0:00:01
   ---------------------------------- ----- 98.6/114.5 MB 56.2 MB/s eta 0:00:01
   ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0 requires torch==2.9.0, but you have torch 2.11.0 which is incompatible.


In [1]:
# Import
import os
import cv2 # used for opening video files and extracting frames
import torch # used for loading the model and performing inference
import numpy as np # used for numerical operations
from pathlib import Path # used for handling file paths
from torch.utils.data import Dataset, DataLoader # used for creating a custom dataset and data loader
from torchvision import transforms # used for image transformations
from sklearn.model_selection import train_test_split # used for splitting the dataset into training and testing sets

In [4]:
DATA_DIR = "C:/Users/tonya/wlasl_top20" # path to the dataset 
NUM_FRAMES = 16 # number of frames to extract from each video
IMG_SIZE = 224 # size to which each frame will be resized
BATCH_SIZE = 8 
NUM_CLASSES = 20
RANDOM_SEED = 42

# get class labels from folder names
CLASSES = sorted(os.listdir(DATA_DIR))
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)} # create a mapping from class names to indices
print(f"class to index mapping: {CLASS_TO_IDX}")

class to index mapping: {'accident': 0, 'bar': 1, 'bed': 2, 'before': 3, 'bowling': 4, 'call': 5, 'candy': 6, 'champion': 7, 'change': 8, 'check': 9, 'cold': 10, 'computer': 11, 'cool': 12, 'cousin': 13, 'deaf': 14, 'delay': 15, 'drink': 16, 'far': 17, 'go': 18, 'help': 19}


In [ ]:
# function to extract frames from a video file

def extract_frames(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # evenly sample frame indices across the video
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
    
    cap.release()
    
    # if video is too short, repeat last frame to pad
    while len(frames) < num_frames:
        frames.append(frames[-1])
    
    return frames

In [ ]:
# sample video to test
sample_class = CLASSES[0]
sample_video = list(Path(f"{DATA_DIR}/{sample_class}").glob("*.mp4"))[0]

frames = extract_frames(str(sample_video))
print(f"Extracted {len(frames)} frames from {sample_video.name}")
print(f"Frame shape: {frames[0].shape}")

Extracted 16 frames from 00623.mp4
Frame shape: (240, 320, 3)


In [ ]:
# TODO: Transform frames (resize, normalize, etc.) and create a custom dataset class to load the data for training.